# Laptop Price Prediction
This notebook demonstrates a complete end-to-end Machine Learning project in Python for **Laptop Price Prediction** using the Kaggle dataset `muhammetvarl/laptop-price`.

## 1. Import required libraries
We will import all necessary libraries for data manipulation, visualization, machine learning, and model saving.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import kagglehub
import warnings
warnings.filterwarnings('ignore')

## 2. Load dataset from CSV file
First, we will download the dataset using `kagglehub` and load it into a Pandas DataFrame.

In [ ]:
# Download latest version of the dataset
path = kagglehub.dataset_download('muhammetvarl/laptop-price')
print('Path to dataset files:', path)

# Load the dataset (adjusting encoding as this dataset usually requires latin-1)
import os
csv_path = os.path.join(path, 'laptop_price.csv')
df = pd.read_csv(csv_path, encoding='latin-1')
df.head()

## 3. Perform complete Exploratory Data Analysis (EDA)
Let's explore the dataset to understand its shape, missing values, duplicates, and statistical properties.

In [ ]:
print('Dataset shape:', df.shape)
print('\nDataset information:')
df.info()
print('\nStatistical summary:')
display(df.describe())
print('\nMissing values:\n', df.isnull().sum())
print('\nDuplicate values:', df.duplicated().sum())
print('\nUnique values in each column:')
for col in df.columns:
    print(f'{col}: {df[col].nunique()} unique values')

## 4. Create professional graphs and visualizations
Visualizing the data distributions and relationships before preprocessing.

In [ ]:
# Distribution Plot of Target Variable (Price_euros)
plt.figure(figsize=(10, 5))
sns.histplot(df['Price_euros'], kde=True, color='blue')
plt.title('Distribution of Laptop Prices (Euros)')
plt.show()

In [ ]:
# Countplot for Company
plt.figure(figsize=(15, 6))
sns.countplot(x='Company', data=df, order=df['Company'].value_counts().index, palette='viridis')
plt.xticks(rotation=45)
plt.title('Count of Laptops by Company')
plt.show()

In [ ]:
# Boxplot for Price vs Company
plt.figure(figsize=(15, 6))
sns.boxplot(x='Company', y='Price_euros', data=df, palette='Set2')
plt.xticks(rotation=45)
plt.title('Price Distribution by Company')
plt.show()

## 5. Perform data preprocessing & 6. Feature Engineering
We need to clean text columns (like Ram and Weight) and convert them to numeric. We also drop the 'laptop_ID' if it exists as it is not a useful feature. Outliers will be handled (e.g., removing extremely expensive laptops if they skew the model too much, but for now we keep them to maintain data integrity).

In [ ]:
# Drop unneeded identifier column if present
if 'laptop_ID' in df.columns:
    df.drop(columns=['laptop_ID'], inplace=True)

# Remove duplicates
df.drop_duplicates(inplace=True)

# Clean 'Ram' column (e.g., '8GB' -> 8)
df['Ram'] = df['Ram'].str.replace('GB', '').astype(int)

# Clean 'Weight' column (e.g., '1.37kg' -> 1.37)
df['Weight'] = df['Weight'].str.replace('kg', '').astype(float)

df.head()

### Feature Engineering and Encoding
Label Encoding categorical features for model training.

In [ ]:
# Label Encoding for categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

df.head()

### Correlation Matrix
Now that all features are numeric, we can visualize the correlation matrix.

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
plt.figure(figsize=(15, 10))
sns.pairplot(df[['Ram', 'Weight', 'Inches', 'Price_euros']])
plt.show()

## 7. Define X features and y target variable
## 8. Split dataset using train_test_split

In [ ]:
X = df.drop(columns=['Price_euros'])
y = df['Price_euros']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Training data shape:', X_train.shape)
print('Testing data shape:', X_test.shape)

## 9. Train ONLY RandomForestRegressor
## 10. Perform Hyperparameter Tuning using GridSearchCV

In [ ]:
rf = RandomForestRegressor(random_state=42)

# Define hyperparameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

print('Best Parameters:', grid_search.best_params_)
best_rf = grid_search.best_estimator_

## 11. Evaluate model

**Important formulas:**
- **MAE** (Mean Absolute Error) = $\frac{1}{n} \sum_{i=1}^{n} |y_{true} - y_{pred}|$
- **MSE** (Mean Squared Error) = $\frac{1}{n} \sum_{i=1}^{n} (y_{true} - y_{pred})^2$
- **RMSE** (Root Mean Squared Error) = $\sqrt{\frac{1}{n} \sum (y_{true} - y_{pred})^2}$
- **R2 Score** = $1 - \frac{\text{Sum Squared Regression (SSR)}}{\text{Total Sum of Squares (SST)}}$

In [ ]:
y_pred = best_rf.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f'MAE: {mae:.2f}')
print(f'MSE: {mse:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'R2 Score: {r2:.4f}')

## 12. Model Result Visualizations
Visualizing Feature Importance, Actual vs Predicted prices, and Residual Errors.

In [ ]:
# Feature Importance Visualization
feature_importances = pd.Series(best_rf.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x=feature_importances, y=feature_importances.index, palette='viridis')
plt.title('Feature Importance')
plt.xlabel('Importance Score')
plt.show()

In [ ]:
# Actual vs Predicted comparison graph
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.5, color='darkorange')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Actual vs Predicted Laptop Prices')
plt.show()

In [ ]:
# Residual error analysis graph
residuals = y_test - y_pred
plt.figure(figsize=(8, 6))
sns.histplot(residuals, kde=True, color='purple')
plt.title('Residual Error Distribution')
plt.xlabel('Error')
plt.show()

## 13. Save trained model using joblib

In [ ]:
# Save the model
joblib.dump(best_rf, 'laptop_price_rf_model.pkl')
print('Model saved successfully!')

# Save the label encoders for future use
joblib.dump(label_encoders, 'label_encoders.pkl')
print('Label Encoders saved successfully!')

## 14. Add prediction example using custom laptop specifications
Demonstrating how to use the trained model for predicting a new laptop's price.

In [ ]:
def predict_laptop_price(custom_data):
    # Load model and encoders
    model = joblib.load('laptop_price_rf_model.pkl')
    encoders = joblib.load('label_encoders.pkl')
    
    # Create DataFrame
    df_custom = pd.DataFrame([custom_data])
    
    # Encode categorical features
    for col in encoders.keys():
        if col in df_custom.columns:
            # Handle unseen labels by mapping them to the first class or a default
            df_custom[col] = df_custom[col].map(lambda s: '\<unknown\>' if s not in encoders[col].classes_ else s)
            encoders[col].classes_ = np.append(encoders[col].classes_, '\<unknown\>')
            df_custom[col] = encoders[col].transform(df_custom[col])
            
    # Predict
    price = model.predict(df_custom)[0]
    return price

# Example specification (using original categorical text)
custom_spec = {
    'Company': 'Apple',
    'Product': 'MacBook Pro',
    'TypeName': 'Ultrabook',
    'Inches': 13.3,
    'ScreenResolution': 'IPS Panel Retina Display 2560x1600',
    'Cpu': 'Intel Core i5 2.3GHz',
    'Ram': 8,
    'Memory': '256GB SSD',
    'Gpu': 'Intel Iris Plus Graphics 640',
    'OpSys': 'macOS',
    'Weight': 1.37
}

predicted_price = predict_laptop_price(custom_spec)
print(f'Predicted Price for Custom Laptop: â‚¬{predicted_price:.2f}')

## 17. Final Conclusion

**Model Performance & Accuracy Discussion:**
The RandomForestRegressor performed well, with a strong $R^2$ score and reasonable RMSE/MAE, indicating it captures the variance in laptop prices effectively. Random Forest is highly effective for tabular data with mixed data types.

**Overfitting vs Underfitting:**
By utilizing `GridSearchCV` to limit `max_depth` and tune `min_samples_split`, we restricted the tree depth to combat overfitting. The residual plot shows errors normally distributed around zero, confirming the model isn't significantly biased (underfitting).

**Future Improvements:**
- **Advanced Feature Engineering**: Extracting exact processor speed, SSD vs HDD memory size, and TouchScreen presence from string columns.
- **Ensemble Methods**: Trying XGBoost or Gradient Boosting to see if we can further reduce the RMSE.
- **Cross Validation**: Implementing robust K-Fold cross-validation on larger hyperparameter grids.